# Multimodel integration (Groq, Google)

In [28]:
import torch

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")


# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")


GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


## Groq integration

In [29]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

GRmodel = init_chat_model( # Method 1: Using init_chat_model with the model string
        "groq:qwen/qwen3.6-27b",
    )

print(GRmodel)

response = GRmodel.invoke("Hello, how are you?")
response.content

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}} client=<groq.resources.chat.completions.Completions object at 0x000001DFB4D07A70> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DFB3AD65A0> model_name='qwen/qwen3.6-27b' model_kwargs={} groq_api_key=SecretStr('**********')


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hello, how are you?"\n   - This is a standard greeting and a common conversational opener.\n   - It\'s polite, casual, and expects a friendly response.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hello"\n   - Question: "how are you?"\n   - Tone: Friendly, conversational\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting\n   - Respond to the question politely and positively\n   - Keep it concise and natural\n   - Optionally, reciprocate by asking how they are doing\n   - Maintain a helpful and approachable tone\n\n4.  **Draft Response (Mental Refinement):**\n   "Hello! I\'m doing well, thank you for asking. How are you doing today? Is there anything I can help you with?"\n\n5.  **Check Against Guidelines:**\n   - Friendly and appropriate? Yes.\n   - Answers the question? Yes.\n   - Invites further conversation? Yes.\n   - Matches AI nature? Yes, acknowledges being an AI im

In [30]:
from langchain_groq import ChatGroq

GRmodel = ChatGroq( # Method 2: Using the ChatGroq class directly
    model="qwen/qwen3.6-27b",
)

response = GRmodel.invoke("Why do parrots have such colorful feathers?")
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots have such colorful feathers?" This is a biological/evolutionary question about parrot plumage coloration.\n\n2.  **Identify Key Concepts**:\n   - Parrot feather coloration\n   - Evolutionary biology\n   - Functions of color in birds\n   - Mechanisms of color production (pigments vs. structural colors)\n   - Ecological/behavioral contexts (mating, communication, camouflage, etc.)\n\n3.  **Brainstorming/Recall Knowledge**:\n   - *Mechanisms*: Parrots get colors from pigments (psittacofulvins for reds/oranges/yellows, carotenoids from diet, melanins for blacks/browns) and structural colors (blue/green from light scattering in feather nanostructures).\n   - *Evolutionary/Functional Reasons*:\n     - Sexual selection: Bright colors signal health, good genes, fitness to potential mates.\n     - Social communication: Species recognition, individual identification, status signaling within

## Gemini integration

In [31]:
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_GENAI_API_KEY")

GOOmodel = init_chat_model( # Method 1: Using init_chat_model with the model string
    "google_genai:gemini-3.5-flash-lite",
    )

response = GOOmodel.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': 'Parrots have such brightly colored feathers primarily for **survival, communication, and evolution**. While humans see these colors as dazzling and exotic, in the parrot’s natural habitat (usually tropical rainforests), those colors actually serve important practical purposes. \n\nHere is a breakdown of why parrots are so colorful:\n\n### 1. Camouflage in the Rainforest Canopy\nTo the human eye, a bright red or green parrot stands out. But in the dappled, sun-dappled light of a tropical rainforest canopy, their colors actually act as camouflage. \n* **Green parrots** (like many amazons and parakeets) blend in seamlessly with the green leaves.\n* **Red, yellow, and blue parrots** (like macaws) break up their body shape against a backdrop of brightly colored tropical flowers, fruits, and shifting shadows. Predators have a hard time spotting them.\n\n### 2. Finding a Mate (Sexual Selection)\nParrots are very picky about who they choose to mate with. Bright, vi

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI

GOOmodel = ChatGoogleGenerativeAI( # Method 2: Using the ChatGoogleGenerativeAI class directly
    model="gemini-3.5-flash-lite",
)

response = GOOmodel.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': 'Parrots are famous for their dazzling reds, blues, yellows, and greens. While they look like flying rainbows just to brighten our world, every vibrant color serves a specific biological purpose, primarily driven by evolution, survival, and reproduction. \n\nHere is why parrots have such colorful feathers:\n\n### 1. Camouflage in the Rainforest\nIt might seem counterintuitive that a bright green or red bird is camouflaged, but in their natural jungle habitat, it makes total sense. \n* **Green parrots:** Tropical rainforests are a wash of deep greens and filtered sunlight. A green parrot sitting in the canopy practically disappears from predators like hawks and eagles. \n* **Colorful accents (Red, Blue, Yellow):** Many parrots have patches of bright colors. When they sit still, they blend in. But when they fly, those flashes of color can actually confuse predators (a phenomenon called "flash coloration") making it hard for a hawk to lock onto a specific targe

# Streaming and Batch

## Streaming

In [33]:
GRmodel.invoke("Write me a 200-word essay on the importance of biodiversity in ecosystems, highlighting the role of different species in maintaining ecological balance and the potential consequences of species loss. Include examples of keystone species and their impact on their habitats.")

AIMessage(content='\n<think>\nThinking Process:\n\n1.  **Deconstruct the Prompt:**\n    *   Topic: Importance of biodiversity in ecosystems.\n    *   Key points: Role of different species in maintaining ecological balance, consequences of species loss.\n    *   Specific requirement: Include examples of keystone species and their impact.\n    *   Length: ~200 words.\n\n2.  **Drafting - Attempt 1 (Mental or rough text):**\n    Biodiversity is the lifeblood of healthy ecosystems, ensuring resilience and stability through complex interdependencies. Every species, from microscopic fungi to apex predators, plays a unique role in nutrient cycling, pollination, and food web dynamics. This variety acts as a buffer against environmental changes, allowing ecosystems to recover from disturbances.\n\n    Keystone species are particularly vital, exerting disproportionate influence on their habitats relative to their abundance. For instance, sea otters control sea urchin populations, preventing overg

In [35]:
for chunk in GOOmodel.stream("Write me a 200-word essay on the importance of biodiversity in ecosystems, highlighting the role of different species in maintaining ecological balance and the potential consequences of species loss. Include examples of keystone species and their impact on their habitats."):
    print(chunk.text, end="|", flush=True) # Print each chunk as it's received flush=True ensures that the output is printed immediately without buffering.

Biodiversity|, the rich variety of life on Earth, is the bedrock of planetary health and ecological| resilience. Every organism—from microscopic soil bacteria to apex predators—plays a distinct, specialized role in maintaining the delicate equilibrium| of natural systems. Plants capture solar energy through photosynthesis, herbivores manage vegetation, carnivores regulate herbivore populations, and decom|posers recycle vital nutrients back into the soil. This intricate web of interdependence ensures that ecosystems can function efficiently, cycle water| and nutrients, and adapt to environmental changes.

The disruption of this balance through species loss can trigger catastrophic cascading effects.| When a single species vanishes, the niches it once filled are left vacant, often leading to overpopulation, resource depletion, or the collapse| of dependent species. This vulnerability is starkly illustrated by keystone species, whose impact on their environment is disproportionately large

## Batch

In [36]:
response = GOOmodel.batch([
    "Write me a 200-word essay on the importance of biodiversity in ecosystems, highlighting the role of different species in maintaining ecological balance and the potential consequences of species loss. Include examples of keystone species and their impact on their habitats.",
    "Explain the concept of climate change and its effects on global weather patterns, ecosystems, and human societies. Discuss the role of greenhouse gases and provide examples of mitigation strategies.",
    "Describe the process of photosynthesis in plants, including the light-dependent and light-independent reactions. Explain how this process contributes to the carbon cycle and supports life on Earth.",
])

for i, res in enumerate(response):
    print(f"Response {i+1}: {res.content}")

Response 1: [{'type': 'text', 'text': 'Biodiversity is the foundation of healthy, functioning ecosystems, representing the incredible variety of life on Earth. Every organism—from microscopic soil bacteria to apex predators—plays a distinct, vital role in maintaining ecological balance. Producers capture solar energy, decomposers recycle nutrients, and consumers regulate populations. Together, this intricate web of interactions ensures ecosystems remain resilient, adaptable, and capable of withstanding environmental shocks like climate change and disease.\n\nWhen species are lost, this delicate equilibrium fractures. The extinction of a single species can trigger a trophic cascade, causing unpredictable domino effects throughout the ecosystem. The removal of keystone species—those that exert a disproportionately large effect on their environment relative to their abundance—vividly illustrates this danger. For instance, sea otters act as a keystone species in marine kelp forests by prey

In [37]:
GOOmodel.batch([
    "Write me a 200-word essay on the importance of biodiversity in ecosystems, highlighting the role of different species in maintaining ecological balance and the potential consequences of species loss. Include examples of keystone species and their impact on their habitats.",
    "Explain the concept of climate change and its effects on global weather patterns, ecosystems, and human societies. Discuss the role of greenhouse gases and provide examples of mitigation strategies.",
    "Describe the process of photosynthesis in plants, including the light-dependent and light-independent reactions. Explain how this process contributes to the carbon cycle and supports life on Earth.",
    ],
    config={
    "max_concurrency": 5, # Limit to 5 parallel requests to avoid overwhelming the model or hitting rate limits
    }
)

[AIMessage(content=[{'type': 'text', 'text': 'Biodiversity, the rich variety of life on Earth, is the fundamental bedrock of resilient and functioning ecosystems. Every organism—from microscopic soil bacteria to apex predators—plays a distinct, irreplaceable role in maintaining ecological balance. Producers capture solar energy, decomposers recycle vital nutrients, and consumers regulate population dynamics. Together, this intricate web of interactions ensures that ecosystems can withstand environmental stressors like climate change, disease, and natural disasters. \n\nThe catastrophic consequences of species loss underscore this interdependence. When a single species vanishes, it can trigger a domino effect known as a trophic cascade, destabilizing the entire habitat. This vulnerability is especially apparent with keystone species, whose impact on their environment is disproportionately large relative to their abundance. \n\nA classic example is the sea otter. By feeding on sea urchin